# Generative Adversarial Network (GAN)
"Generative Adversarial Nets" (Goodfellow et al., 2014)

**Idea:** train two networks against each other. A **generator** turns random noise into fake images; a **discriminator** tries to tell real images from fakes. The generator's only training signal is the discriminator's gradient: "change the image this way and I would be more convinced". No likelihood, no reconstruction loss.

### GAN vs VAE vs diffusion
| | VAE (`vae.ipynb`) | GAN | Diffusion (`diffusion.ipynb`) |
|---|---|---|---|
| Networks | encoder + decoder | **generator + discriminator** | one denoiser |
| Loss | reconstruction + KL (ELBO) | **adversarial** (a learned loss) | noise-prediction MSE |
| Objective is | a fixed function to minimize | a two-player **game** (saddle point) | a fixed function to minimize |
| Sampling | 1 decoder pass | 1 generator pass | hundreds of denoiser passes |
| Sample quality | blurry | sharp | sharp |
| Mode coverage | good | can drop modes (**mode collapse**) | good |
| Likelihood | lower bound | none | lower bound |
| Training | stable | unstable, sensitive to hyperparameters | stable |

Here: the original fully-connected GAN on MNIST, with the non-saturating generator loss the paper recommends, and a check for mode collapse using a separately trained digit classifier.

Shape flow: `z (B, 100) -> G -> fake (B, 784) in [-1, 1]` and `real or fake (B, 784) -> D -> logit (B,)` ("how real does this look")

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - MNIST
60k digits, 28x28, flattened to 784 and scaled to $[-1, 1]$ to match the generator's `tanh` output. Labels are **not** used by the GAN (unconditional generation); they are kept only to train the classifier used for evaluation in §6.

In [ ]:
train_ds = datasets.MNIST("../data", train=True, download=True)
X = (train_ds.data.float() / 127.5 - 1).reshape(-1, 784).to(device)   # (60000, 784) in [-1, 1]
y = train_ds.targets.to(device)
print(X.shape, X.min().item(), X.max().item())

## 2. Generator and discriminator
Both are MLPs.
- **G**: noise $z \in \mathbb{R}^{100} \to$ image. Final `tanh` keeps pixels in $[-1, 1]$.
- **D**: image $\to$ one logit. `LeakyReLU` keeps a gradient for negative inputs (D's gradient is G's training signal, so it must not die), and dropout stops D from winning too easily.

### Formula: The two players
$$G_\theta : z \mapsto x_{fake}, \quad z \sim \mathcal{N}(0, I) \qquad\qquad D_\phi : x \mapsto D(x) = \sigma(\text{logit}) \in (0, 1)$$
- $z \in \mathbb{R}^{100}$: latent noise; the generator defines a distribution $p_g$ over images implicitly (we can sample from it, but not evaluate it)
- $D(x)$: the discriminator's probability that $x$ is real

In [ ]:
latent_dim = 100

class Generator(nn.Module):
    # z (B, 100) -> image (B, 784) in [-1, 1]
    def __init__(self, latent_dim=100, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, 2 * hidden), nn.LeakyReLU(0.2),
            nn.Linear(2 * hidden, 4 * hidden), nn.LeakyReLU(0.2),
            nn.Linear(4 * hidden, 784), nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    # image (B, 784) -> logit (B,)
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 4 * hidden), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(4 * hidden, 2 * hidden), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(2 * hidden, hidden), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

G, D = Generator().to(device), Discriminator().to(device)
print(f"G params: {sum(p.numel() for p in G.parameters()) / 1e6:.2f}M | D params: {sum(p.numel() for p in D.parameters()) / 1e6:.2f}M")
z = torch.randn(4, latent_dim, device=device)
print(G(z).shape, D(G(z)).shape)

## 3. The game
### Formula: Minimax objective
$$\min_G \max_D\ V(D, G) = \mathbb{E}_{x \sim p_{data}}\big[\log D(x)\big] + \mathbb{E}_{z \sim p_z}\big[\log\big(1 - D(G(z))\big)\big]$$
- D maximizes: push $D(x) \to 1$ on real images and $D(G(z)) \to 0$ on fakes. This is binary cross-entropy with labels real = 1, fake = 0
- G minimizes: push $D(G(z)) \to 1$

### Formula: Optimal discriminator and what G then minimizes
$$D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}, \qquad V(D^*, G) = -\log 4 + 2\, \text{JSD}\big(p_{data}\ \|\ p_g\big)$$
- for a fixed G, the best D is the density ratio above
- plugging it in, G minimizes the **Jensen-Shannon divergence** between real and generated distributions
- at the optimum $p_g = p_{data}$: $D^* = 1/2$ everywhere, D's loss is $\log 4 \approx 1.386$ and G's is $\log 2 \approx 0.693$

### Formula: Non-saturating generator loss (used in practice)
$$\mathcal{L}_G = -\mathbb{E}_{z}\big[\log D(G(z))\big] \qquad \text{instead of} \qquad \mathbb{E}_{z}\big[\log\big(1 - D(G(z))\big)\big]$$
- early in training D rejects fakes confidently, $D(G(z)) \approx 0$. There $\log(1 - D)$ is flat (gradient $\approx 0$): G learns nothing exactly when it is worst
- $-\log D$ has the same fixed point but a **large** gradient when $D(G(z)) \approx 0$
- in code: binary cross-entropy on fakes with the label flipped to "real"

### Formula: Discriminator loss
$$\mathcal{L}_D = -\mathbb{E}_{x}\big[\log D(x)\big] - \mathbb{E}_{z}\big[\log\big(1 - D(G(z))\big)\big]$$
- the fake batch is **detached**: D's update must not push gradients into G

In [ ]:
# the saturation problem in numbers: gradient of each generator loss w.r.t. D's logit on a fake
logit = torch.tensor([-6.0, -3.0, 0.0, 3.0], requires_grad=True)       # D(G(z)) = sigmoid(logit)
saturating = torch.log(1 - torch.sigmoid(logit))                         # log(1 - D)
non_saturating = -F.logsigmoid(logit)                                    # -log D
g_sat = torch.autograd.grad(saturating.sum(), logit)[0]
g_ns = torch.autograd.grad(non_saturating.sum(), logit)[0]
for l, a, b in zip(logit.tolist(), g_sat.tolist(), g_ns.tolist()):
    print(f"D(G(z)) = {torch.sigmoid(torch.tensor(l)).item():.3f} | d/dlogit log(1 - D) = {a:+.3f} | d/dlogit -log D = {b:+.3f}")

## 4. Training
Each step:
1. **D step**: one real batch (label 1) + one fake batch (label 0, detached).
2. **G step**: a fresh fake batch, labelled 1, gradients flow through D into G. Only G's optimizer steps.

Adam with $\beta_1 = 0.5$ (from DCGAN: the default 0.9 momentum makes the two-player dynamics oscillate), learning rate $2 \times 10^{-4}$ for both.

The losses do **not** go down like in ordinary training. They measure how the two players are doing relative to each other; a healthy run hovers near the equilibrium values. Progress is judged by looking at samples from a **fixed** set of noise vectors.

In [ ]:
epochs = 50
batch_size = 128
lr = 2e-4

opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

fixed_z = torch.randn(16, latent_dim, device=device)     # same noise every time, to watch samples evolve
snapshots, history = {}, []

for epoch in range(1, epochs + 1):
    perm = torch.randperm(len(X), device=device)
    d_loss_sum = g_loss_sum = d_real_sum = d_fake_sum = 0.0
    steps = 0
    for i in range(0, len(X) - batch_size + 1, batch_size):
        real = X[perm[i:i + batch_size]]                                # (B, 784)
        ones, zeros = torch.ones(batch_size, device=device), torch.zeros(batch_size, device=device)

        # --- discriminator step ---
        z = torch.randn(batch_size, latent_dim, device=device)          # (B, 100)
        fake = G(z).detach()                                            # (B, 784), no gradient into G
        real_logit, fake_logit = D(real), D(fake)                       # (B,), (B,)
        d_loss = F.binary_cross_entropy_with_logits(real_logit, ones) + F.binary_cross_entropy_with_logits(fake_logit, zeros)
        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        # --- generator step (non-saturating) ---
        z = torch.randn(batch_size, latent_dim, device=device)
        g_loss = F.binary_cross_entropy_with_logits(D(G(z)), ones)      # fakes labelled "real": -log D(G(z))
        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

        d_loss_sum += d_loss.item(); g_loss_sum += g_loss.item()
        d_real_sum += torch.sigmoid(real_logit).mean().item(); d_fake_sum += torch.sigmoid(fake_logit).mean().item()
        steps += 1

    history.append((epoch, d_loss_sum / steps, g_loss_sum / steps, d_real_sum / steps, d_fake_sum / steps))
    if epoch in (1, 5, 20, epochs):
        G.eval()
        with torch.no_grad():
            snapshots[epoch] = G(fixed_z).cpu()
        G.train()
    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:2d} | D loss {history[-1][1]:.3f} | G loss {history[-1][2]:.3f} | D(real) {history[-1][3]:.3f} | D(fake) {history[-1][4]:.3f}")

## 5. Results
Left: the two losses with their equilibrium values (dashed). Right: D's average output on real and fake batches; both drift toward 0.5 as fakes get harder to tell apart.

Below: the same 16 noise vectors decoded at different epochs.

In [ ]:
ep, dl, gl, dr, df = zip(*history)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(ep, dl, label="D loss"); ax[0].plot(ep, gl, label="G loss")
ax[0].axhline(math.log(4), ls="--", c="C0", lw=0.8); ax[0].axhline(math.log(2), ls="--", c="C1", lw=0.8)
ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(ep, dr, label="D(real)"); ax[1].plot(ep, df, label="D(fake)"); ax[1].axhline(0.5, ls="--", c="gray", lw=0.8)
ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

fig, axes = plt.subplots(len(snapshots), 16, figsize=(16, len(snapshots) * 1.05))
for row, (epoch, imgs) in zip(axes, snapshots.items()):
    for a, im in zip(row, imgs):
        a.imshow(im.reshape(28, 28), cmap="gray"); a.axis("off")
    row[0].set_title(f"epoch {epoch}", fontsize=8, loc="left")
plt.show()

## 6. Mode collapse check
A GAN can fool D while producing only *some* kinds of digits: nothing in the objective forces G to cover every mode of the data. There is no likelihood to detect this, so we use an outside judge: a small classifier trained on real MNIST labels the generated digits, and we look at the class histogram (real data is close to 10% each).

(The Inception Score and FID used for image GANs are the same idea with an ImageNet classifier's outputs / features.)

In [ ]:
torch.manual_seed(0)
clf = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
for epoch in range(3):
    perm = torch.randperm(len(X), device=device)
    for i in range(0, len(X), 128):
        ix = perm[i:i + 128]
        loss = F.cross_entropy(clf(X[ix]), y[ix])
        opt.zero_grad(); loss.backward(); opt.step()

test_ds = datasets.MNIST("../data", train=False, download=True)
Xte = (test_ds.data.float() / 127.5 - 1).reshape(-1, 784).to(device)
print(f"classifier test acc: {(clf(Xte).argmax(-1).cpu() == test_ds.targets).float().mean().item():.4f}")

G.eval()
with torch.no_grad():
    samples = G(torch.randn(10000, latent_dim, device=device))           # (10000, 784)
    probs = clf(samples).softmax(-1)                                     # (10000, 10)
pred = probs.argmax(-1)
hist = torch.bincount(pred, minlength=10).float() / len(pred)
real_hist = torch.bincount(y, minlength=10).float() / len(y)
print("generated class shares:", [round(v, 3) for v in hist.tolist()])
print("real class shares     :", [round(v, 3) for v in real_hist.tolist()])
print(f"mean classifier confidence on generated digits: {probs.max(-1).values.mean().item():.3f}")

plt.figure(figsize=(6, 3))
plt.bar(torch.arange(10) - 0.2, real_hist.cpu(), width=0.4, label="real")
plt.bar(torch.arange(10) + 0.2, hist.cpu(), width=0.4, label="generated")
plt.xticks(range(10)); plt.xlabel("digit (as judged by the classifier)"); plt.ylabel("share"); plt.legend(); plt.show()

## 7. Walking through latent space
Interpolate between two noise vectors and generate at each step. A generator that has learned a smooth map (rather than memorizing training images) morphs one digit into another through digit-like shapes.

In [ ]:
with torch.no_grad():
    rows = []
    for _ in range(3):
        a, b = torch.randn(2, latent_dim, device=device)
        w = torch.linspace(0, 1, 12, device=device)[:, None]             # (12, 1)
        rows.append(G((1 - w) * a + w * b).cpu())                        # (12, 784)
fig, axes = plt.subplots(3, 12, figsize=(12, 3.2))
for row, imgs in zip(axes, rows):
    for a, im in zip(row, imgs):
        a.imshow(im.reshape(28, 28), cmap="gray"); a.axis("off")
plt.show()

## Notes
- The discriminator is a **learned loss function**. A pixel-wise loss (VAE) averages over plausible outputs and blurs; D instead penalizes anything that does not look like data, including blur, so GAN samples are sharp.
- Why training is fragile: it is not minimization but a search for a saddle point, and gradient descent on both players can orbit instead of converge. If D gets too good its gradient vanishes (the JSD is flat when the two distributions do not overlap); if D is too weak its gradient is uninformative.
- Fixes that followed: **DCGAN** (conv architecture + BatchNorm, the recipe that made GANs work on images), **WGAN / WGAN-GP** (Wasserstein distance with a Lipschitz critic, gradients that do not vanish), **spectral norm**, the hinge loss, and **StyleGAN** (state of the art for faces).
- **Conditional GANs** feed the label (or a text / image) to both G and D; pix2pix and CycleGAN do image-to-image translation this way.
- Diffusion models replaced GANs for general image generation (stable training, full mode coverage), but GANs remain in use where one-step sampling matters, and the **adversarial loss** lives on inside other systems: the autoencoders of latent diffusion (`dit.ipynb`) and neural audio codecs are trained with a discriminator, and diffusion distillation into 1-4 step generators uses one too.
- With 100 latent dims and an MLP, expect digits that are recognizable but rougher than `diffusion.ipynb`'s.